# Week 3 — Dimensional Modeling Exploration

This notebook validates the proposed star-schema design for the Special Olympics ETL pipeline.
We check dimension cardinalities, test event-code extraction, and confirm grain decisions
before committing to the formal model documentation.

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))

from src.core import RAW_DIR
from src.utils.data_loader import DataLoader
import pandas as pd

loader = DataLoader(RAW_DIR)
all_results = loader.load_all_results()
cert_df = loader.load_certifications()
clubs_df = loader.load_clubs()

print(f"Results rows:        {len(all_results):,}")
print(f"Certifications rows: {len(cert_df):,}")
print(f"Clubs rows:          {len(clubs_df):,}")

## 1. Dimension Cardinalities

Before finalizing the star schema we need to know how large each dimension will be.
Small dimensions (< 1 000 rows) are great for Power BI slicers; large ones may need hierarchies.

In [ ]:
n_athletes = cert_df["Code"].nunique()
n_clubs = len(clubs_df)
n_sports = all_results["Sport"].nunique()
n_events_raw = all_results["Event"].nunique()
n_years = all_results["Year"].nunique()

summary = pd.DataFrame({
    "Dimension": ["dim_athlete", "dim_geography", "dim_sport", "dim_event (raw)", "dim_time"],
    "Row Count": [n_athletes, n_clubs + 1, n_sports, n_events_raw, n_years],
    "Source": ["Certifications.Code", "Clubs + 1 Unknown", "Results.Sport", "Results.Event", "Filename year"],
})
summary

### Observations

- **dim_athlete** (~20 221 rows) is the largest dimension. This includes non-competing coaches/staff
  from Certifications. Only ~7 607 athletes actually appear in Results.
- **dim_geography** (437 rows including Unknown) is small — perfect for slicer/filter use.
- **dim_sport** (23 rows) is tiny.
- **dim_event** (377 raw names) is inflated by bilingual naming. We'll reduce this with event-code extraction.
- **dim_time** (9 rows) is trivially small.

## 2. Event Code Extraction

Event names are bilingual and inconsistent across years (375+ variants).
Each event has a **code prefix** (e.g., `AT17`, `AQ25`, `CY05`) that stays stable.
We test whether extracting this prefix produces a reliable, smaller set of canonical events.

In [ ]:
events = all_results["Event"].dropna().unique()

def extract_event_code(event_name: str) -> str:
    """Extract the alphanumeric event code prefix from an event name.
    
    Examples:
        'AT17 - Softbalwerpen/Lancement softball' -> 'AT17'
        'AQ25 - 50m Vrije slag' -> 'AQ25'
        'AG - Niveau A' -> 'AG'
        'APA - 40m Sprint' -> 'APA'
    """
    name = str(event_name).strip()
    # Match letter prefix + optional digits before ' - ' separator
    m = re.match(r"^([A-Za-z]{2,5}\d*)\s*-", name)
    if m:
        return m.group(1).upper()
    # Fallback: some events lack the ' - ' separator (edge cases)
    m = re.match(r"^([A-Za-z]{2,5}\d*)", name)
    if m:
        return m.group(1).upper()
    return "UNKNOWN"

codes = [extract_event_code(e) for e in events]
unique_codes = sorted(set(codes))
print(f"Raw event names: {len(events)}")
print(f"Unique event codes after extraction: {len(unique_codes)}")
print(f"\nReduction: {len(events)} -> {len(unique_codes)} ({100 - len(unique_codes)/len(events)*100:.0f}% fewer)")

In [ ]:
# Show event codes with multiple name variants (the ones normalization collapses)
code_to_names = {}
for e in events:
    c = extract_event_code(e)
    code_to_names.setdefault(c, []).append(e)

multi_variant = {k: v for k, v in code_to_names.items() if len(v) > 1}
print(f"Event codes with multiple name variants: {len(multi_variant)}")
print()
for code in sorted(multi_variant)[:10]:
    names = multi_variant[code]
    print(f"  {code} ({len(names)} variants):")
    for n in names[:3]:
        print(f"    - {n}")
    if len(names) > 3:
        print(f"    ... and {len(names) - 3} more")

### Edge Cases — Events Without Standard Code Prefixes

Some event names don't follow the `CODE - Description` pattern. These are mostly
finals, divisioning rounds, and APA sub-events with non-standard naming.

In [ ]:
edge_cases = [(e, extract_event_code(e)) for e in sorted(events) 
              if extract_event_code(e) in (
                  "FINAL", "FINALS", "DIVIS", "EVENW", "HINDE", 
                  "KEGEL", "PETAN", "SPRIN", "UNIHO", "UNKNOWN"
              )]
print(f"Edge case events: {len(edge_cases)}")
for name, code in edge_cases:
    print(f"  [{code}] {name}")

### Decision

The event code extraction reduces 377 raw names to ~182 codes — a 52% reduction.
Edge cases (~12 events) are special rounds (finals, divisioning) or APA sub-activities
written in full Dutch/French without a code prefix. These will be assigned codes during ETL
based on their parent sport (e.g., `FO-FINAL`, `APA-PETANQUE`).

**Conclusion:** Use `event_code` as the stable key in `dim_event`. Store one canonical
event name per code (pick the most recent year's name or the most common variant).

## 3. Fact Table Grain Validation

The proposed grain for `fact_results` is **one row per athlete × event × year** (final round only).
Let's validate what deduplication looks like.

In [ ]:
raw_count = len(all_results)
deduped_count = all_results.drop_duplicates(["Code", "Event", "Sport", "Year"]).shape[0]
dup_pct = (1 - deduped_count / raw_count) * 100

print(f"Raw result rows:             {raw_count:>10,}")
print(f"After dedup (Code+Event+Sport+Year): {deduped_count:>10,}")
print(f"Rows removed:                {raw_count - deduped_count:>10,} ({dup_pct:.1f}%)")
print()
print("Per-year breakdown:")
for year in sorted(all_results["Year"].unique()):
    yr = all_results[all_results["Year"] == year]
    raw = len(yr)
    dedup = yr.drop_duplicates(["Code", "Event", "Sport"]).shape[0]
    print(f"  {year}: {raw:>6,} raw -> {dedup:>6,} deduped ({(1-dedup/raw)*100:.0f}% removed)")

### Observation

~35% of rows are multi-round duplicates. The deduplication strategy (keep final/best round)
will reduce `fact_results` from ~112K to ~73K rows — a manageable size for Power BI.

The ETL will need to pick the "best" row per (Code, Event, Sport, Year) group — likely
the one with a non-null Place value, or the one from the `Summary (all)` column.

## 4. Gender Standardization Check

Certifications uses `M`/`F`/`U`, while Results uses `Male`/`Female`/`Unknown`.
The ETL must standardize these.

In [ ]:
print("Certifications Gender values:")
print(cert_df["Gender"].value_counts(dropna=False).to_string())
print()
print("Results Gender values:")
print(all_results["Gender"].value_counts(dropna=False).to_string())

**Decision:** Standardize to `M`/`F`/`U` in `dim_athlete` (shorter, consistent with Certifications master).
Map `Male`→`M`, `Female`→`F`, `Unknown`→`U` in Results during ETL.

## 5. Sport Naming: Aquatics/Swimming vs Swimming

The data exploration flagged that `Aquatics/Swimming` (2015–2023) was replaced by
`Swimming` (2024–2025). Let's verify whether they are truly different.

In [ ]:
for sport in ["Aquatics/Swimming", "Swimming"]:
    subset = all_results[all_results["Sport"] == sport]
    years = sorted(subset["Year"].unique())
    event_prefixes = sorted(set(
        extract_event_code(e) for e in subset["Event"].dropna().unique()
    ))
    print(f"{sport}:")
    print(f"  Years: {years}")
    print(f"  Event prefixes: {event_prefixes}")
    print(f"  Row count: {len(subset):,}")
    print()

### Decision

- **Aquatics/Swimming** uses `AQ` event codes (25m/50m/100m pool events) — present 2015–2023.
- **Swimming** uses `SW` event codes (33m/66m/99m pool events) — present in all years including 2024–2025.

These are **genuinely different competition categories** with different pool lengths.
However, `Aquatics/Swimming` disappears in 2024–2025 while `Swimming` continues.
In `dim_sport`, keep them as **two separate sports** to preserve the distinction.

If `Aquatics/Swimming` was simply renamed to include its events under `Swimming` in later years,
the ETL can revisit this. For now, the event code prefix (`AQ` vs `SW`) keeps them apart.

## 6. Summary — Confirmed Design Decisions

| Decision | Rationale |
|----------|----------|
| `dim_athlete` includes all person types | Allows coach/volunteer analysis in Power BI via filter |
| `dim_geography` has an Unknown row (key = -1) | Captures ~15% of Results with unmatched clubs |
| `dim_event` uses extracted event codes | Reduces 377 names → ~182 stable codes |
| `dim_time` is year-grain only | No specific dates in data; COVID period labels added |
| Aquatics/Swimming ≠ Swimming | Different pool formats (AQ vs SW event codes) |
| Gender standardized to M/F/U | Certifications master format; Results uses Male/Female |
| `fact_results` grain: 1 row per athlete×event×year | ~35% dedup from multi-round data |
| `fact_participation` as separate table | Simplifies DAX; one row per athlete×club×year |